In [ ]:
# ============================================================
# 05_LSTM.ipynb
# Section 1 — Setup, Paths, Data Loading, GPU Check
# Purpose: Load preprocessed FD001 data, import TF/Keras,
#          verify GPU, prepare for sequence modeling.
# ============================================================

# --- 1a. Mount Drive ---

# --- 1b. Canonical paths (same as all notebooks) ---
import os

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
DATA_DIR      = os.path.join(PROJECT_ROOT, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
FIGURES_DIR   = os.path.join(PROJECT_ROOT, "figures")
EXPERIMENT_DIR = os.path.join(PROJECT_ROOT, "experiments")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")

import sys
sys.path.insert(0, SRC_DIR)

# --- 1c. Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_absolute_error, mean_squared_error
from preprocessing import FINAL_FEATURES_FD001

print(f"TensorFlow version: {tf.__version__}")

# --- 1d. GPU check ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✓ GPU available: {gpus[0].name}")
    gpu_available = True
else:
    print("⚠ NO GPU FOUND — go to Runtime → Change runtime type → GPU")
    print("  Training will be very slow on CPU.")
    gpu_available = False

# --- 1e. Reproducibility: fix all seeds ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)
print(f"\nRandom seeds fixed: {SEED}")
print("Note: GPU training has minor non-determinism even with fixed seeds;")
print("results may vary by small amounts between runs. This is documented.")

# --- 1f. Load preprocessed data ---
train = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))

print("\n=== Data loaded from 02_Preprocessing output ===\n")
print(f"Train shape:      {train.shape}")
print(f"Test-full shape:  {test_full.shape}")
print(f"Test-last shape:  {test_last.shape}")

# --- 1g. Verification ---
assert train.shape[0] == 20631
assert test_full.shape[0] == 13096
assert test_last.shape[0] == 100
assert all(f in train.columns for f in FINAL_FEATURES_FD001)
assert "RUL_linear" in train.columns
assert "RUL_piecewise125" in train.columns
assert "RUL_true" in test_last.columns

print("\n✓ All checks passed. Ready for sequence construction.")

In [ ]:
# ============================================================
# 05_LSTM.ipynb
# Section 2 — Metrics + Sequence Construction
# Purpose: (a) identical metric functions as Notebooks 03/04;
#          (b) sliding-window builders for train/val/test with
#          zero pre-padding and verified alignment.
# ============================================================

# --- 2a. Metrics (identical to Notebooks 03/04) ---
def nasa_score(y_true, y_pred):
    """PHM08 asymmetric scoring function (Saxena et al. 2008)."""
    d = y_pred - y_true
    scores = np.where(d < 0,
                      np.exp(-d / 13.0) - 1.0,
                      np.exp( d / 10.0) - 1.0)
    return float(np.sum(scores))

def evaluate_model(y_true, y_pred, model_name="Model", label_name="label"):
    """Compute MAE, RMSE, NASA Score; print summary; return dict."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)
    print(f"--- {model_name} | {label_name} ---")
    print(f"  MAE:        {mae:.2f} cycles")
    print(f"  RMSE:       {rmse:.2f} cycles")
    print(f"  NASA Score: {ns:.2f}")
    print()
    return {"model": model_name, "rul_label": label_name,
            "mae": round(mae, 2), "rmse": round(rmse, 2),
            "nasa_score": round(ns, 2)}

perfect = nasa_score(np.array([100, 50, 25]), np.array([100, 50, 25]))
assert perfect == 0.0
assert nasa_score(np.array([100]), np.array([110])) > nasa_score(np.array([100]), np.array([90]))
print("✓ Metrics defined and verified (identical to Notebooks 03/04).\n")

# --- 2b. Sequence builders ---
N_FEATURES = len(FINAL_FEATURES_FD001)   # 13

def build_train_sequences(df, window, label_col):
    """
    Sliding windows over every engine trajectory.
    Window [c-W+1 ... c] -> label = RUL at cycle c (window's last cycle).
    Early cycles with insufficient history are zero-PRE-padded.
    Returns X (n, window, 13) and y (n,).
    """
    X_list, y_list = [], []
    for unit, grp in df.groupby("unit"):
        grp = grp.sort_values("cycle")
        feats = grp[FINAL_FEATURES_FD001].values          # (L, 13)
        labels = grp[label_col].values                    # (L,)
        L = len(grp)
        for end in range(L):                              # window ends at index `end`
            start = max(0, end - window + 1)
            seq = feats[start:end + 1]                    # (<=window, 13)
            if len(seq) < window:                         # pre-pad with zeros
                pad = np.zeros((window - len(seq), N_FEATURES))
                seq = np.vstack([pad, seq])
            X_list.append(seq)
            y_list.append(labels[end])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

def build_last_window_sequences(df_full, window, units_order):
    """
    One window per engine: the LAST `window` cycles (zero-pre-padded
    if the engine is shorter). Prediction at the last observed cycle.
    `units_order` fixes the engine order (must match the y_true order).
    Returns X (n_engines, window, 13).
    """
    X_list = []
    for unit in units_order:
        grp = df_full[df_full["unit"] == unit].sort_values("cycle")
        feats = grp[FINAL_FEATURES_FD001].values
        seq = feats[-window:]                             # last <=window cycles
        if len(seq) < window:
            pad = np.zeros((window - len(seq), N_FEATURES))
            seq = np.vstack([pad, seq])
        X_list.append(seq)
    return np.array(X_list, dtype=np.float32)

print("✓ Sequence builders defined.\n")

# --- 2c. Verify with a small example (window=30, piecewise label) ---
W_CHECK = 30
X_check, y_check = build_train_sequences(train, W_CHECK, "RUL_piecewise125")
print(f"Train sequences (window={W_CHECK}): X {X_check.shape}, y {y_check.shape}")
# Every cycle of every engine yields exactly one sample:
assert X_check.shape[0] == len(train), "Expected one sample per training row"
assert X_check.shape[1:] == (W_CHECK, N_FEATURES)

# Alignment spot-check: engine 1, its 50th cycle (index 49).
# The sample index for engine 1 cycle 50 is 49 (engine 1 comes first).
e1 = train[train["unit"] == 1].sort_values("cycle")
manual_label = e1["RUL_piecewise125"].values[49]
assert y_check[49] == manual_label, "Label misaligned!"
# The window's last row must equal engine 1's features at cycle 50:
manual_feats = e1[FINAL_FEATURES_FD001].values[49]
assert np.allclose(X_check[49, -1, :], manual_feats), "Window misaligned!"
print("✓ Alignment verified: window's last cycle ↔ label at that cycle.")

# Padding spot-check: engine 1's first sample (cycle 1) must be
# zero-padded in its first 29 rows:
assert np.allclose(X_check[0, :W_CHECK-1, :], 0.0), "Padding wrong!"
assert not np.allclose(X_check[0, -1, :], 0.0)
print("✓ Zero pre-padding verified for early cycles.")

# --- 2d. Test windows (order must match test_last!) ---
units_order = test_last["unit"].values                    # canonical order
X_test_check = build_last_window_sequences(test_full, W_CHECK, units_order)
y_true = test_last["RUL_true"].values
print(f"\nTest sequences (window={W_CHECK}): X {X_test_check.shape}")
assert X_test_check.shape == (100, W_CHECK, N_FEATURES)

# Alignment spot-check: last row of engine's window == its last-cycle
# features from test_last (they came from the same processed source).
i0_unit = units_order[0]
row_from_last = test_last[test_last["unit"] == i0_unit][FINAL_FEATURES_FD001].values[0]
assert np.allclose(X_test_check[0, -1, :], row_from_last), "Test window misaligned!"
print("✓ Test windows aligned with test_last (prediction at last cycle).")

print("\n✓ Section 2 complete: metrics + verified sequence machinery.")

In [ ]:
# ============================================================
# 05_LSTM.ipynb
# Section 3 — Validation Split, Bounded Sweep, Final Training
# Purpose: Same protocol as XGBoost (Notebook 04): engine-level
#          20-engine validation (seed 42), truncation (seed 123),
#          6-config sweep, selection on val RMSE, final train on
#          all engines, single test evaluation per label.
# ============================================================

# --- 3a. Engine-level validation split (IDENTICAL to Notebook 04) ---
#rng = np.random.RandomState(42)
#all_units = train["unit"].unique()
#val_units = rng.choice(all_units, size=20, replace=False)
#train_units = np.array([u for u in all_units if u not in val_units])

#train_sub = train[train["unit"].isin(train_units)]
#val_sub   = train[train["unit"].isin(val_units)]

#print(f"Training engines:   {len(train_units)}  ({len(train_sub)} rows)")
#print(f"Validation engines: {len(val_units)}")
#print(f"Validation units:   {sorted(val_units.tolist())}")
#print("(Must match Notebook 04: [1, 5, 11, 13, 19, 23, 31, 32, 34, 40, "
  #    "45, 46, 54, 71, 74, 77, 78, 81, 84, 91])\n")

# --- 3b. Truncated validation engines (IDENTICAL protocol, seed 123) ---
#def make_truncation_points(val_sub, rng, min_keep=30):
 #   """Random truncation cycle per validation engine (same as NB04)."""
  #  cuts = {}
   # for unit, grp in val_sub.groupby("unit"):
    #    c_max = grp["cycle"].max()
       # cuts[unit] = (rng.randint(min_keep, c_max), c_max)
  ##  return cuts

#cuts = make_truncation_points(val_sub, np.random.RandomState(123))
#y_val_true = np.array([c_max - cut for (cut, c_max) in
 #                      [cuts[u] for u in sorted(cuts)]], dtype=float)
#val_units_sorted = sorted(cuts)
#print(f"Truncated validation: {len(cuts)} engines, "
#      f"RUL range {y_val_true.min():.0f}–{y_val_true.max():.0f}\n")

#def build_val_windows(val_sub, cuts, window, units_order):
 #   """Last `window` cycles up to each engine's truncation cycle."""
  #  X_list = []
   # for unit in units_order:
     #   cut, _ = cuts[unit]
     #   grp = val_sub[(val_sub["unit"] == unit) &
     #                 (val_sub["cycle"] <= cut)].sort_values("cycle")
      #  feats = grp[FINAL_FEATURES_FD001].values
      #  seq = feats[-window:]
      #  if len(seq) < window:
       #     seq = np.vstack([np.zeros((window - len(seq), N_FEATURES)), seq])
       # X_list.append(seq)
 #   return np.array(X_list, dtype=np.float32)

# --- 3c. Model builder ---
#def build_lstm(window, units1, units2, lr=0.001, dropout=0.2):
    #"""Masking -> LSTM(units1) -> Dropout -> LSTM(units2) -> Dropout
    #   -> Dense(1). Contracting form."""
   # model = keras.Sequential([
     #   layers.Input(shape=(window, N_FEATURES)),
      #  layers.Masking(mask_value=0.0),
      #  layers.LSTM(units1, return_sequences=True),
     #   layers.Dropout(dropout),
       # layers.LSTM(units2),
       # layers.Dropout(dropout),
       # layers.Dense(1),
    #])
  #  model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
      #            loss="mse")
    #return model

# --- 3d. The sweep: 3 windows x 2 architectures ---
#SWEEP = [
 #   {"window": 15, "units1": 64,  "units2": 32},
#    {"window": 30, "units1": 64,  "units2": 32},
  #  {"window": 50, "units1": 64,  "units2": 32},
  #  {"window": 15, "units1": 128, "units2": 64},
   ## {"window": 30, "units1": 128, "units2": 64},
  ##  {"window": 50, "units1": 128, "units2": 64},
#]
#EPOCHS, BATCH, PATIENCE = 50, 64, 5

#def run_lstm_sweep(label_col):
    #"""Train each config on train_sub; select on truncated-val RMSE."""
    #results = []
    # Pre-build sequences once per window size (expensive step)
   # seq_cache = {}
   # for w in sorted({cfg["window"] for cfg in SWEEP}):
      #  X_tr, y_tr = build_train_sequences(train_sub, w, label_col)
      #  X_v = build_val_windows(val_sub, cuts, w, val_units_sorted)
      #  seq_cache[w] = (X_tr, y_tr, X_v)
     #  print(f"  [window={w}] train sequences: {X_tr.shape}")
   # print()
   # for cfg in SWEEP:
       # w = cfg["window"]
       # X_tr, y_tr, X_v = seq_cache[w]
      #  keras.utils.set_random_seed(SEED)          # reset seed per config
      #  model = build_lstm(w, cfg["units1"], cfg["units2"])
      #  es = keras.callbacks.EarlyStopping(monitor="val_loss",
                                #           patience=PATIENCE,
                                  #         restore_best_weights=True)
     #   t0 = time.time()
     #   hist = model.fit(X_tr, y_tr,
              #           validation_split=0.1,      # internal early stopping
              #           epochs=EPOCHS, batch_size=BATCH,
             #            callbacks=[es], verbose=0)
      #  t_train = time.time() - t0
      #  pred_v = model.predict(X_v, verbose=0).ravel()
      #  n_ep = len(hist.history["loss"])
      #  results.append({**cfg, "val_rmse": rmse_v,
             #           "epochs_run": n_ep, "train_time_s": t_train})
     #   print(f"  window={w:>2}, {cfg['units1']}→{cfg['units2']:<3} "
          #    f"->  val RMSE {rmse_v:6.2f}  ({n_ep} epochs, {t_train:.0f}s)")
   # return results

#print("=== Sweep — piecewise125 label ===")
#sweep_pw = run_lstm_sweep("RUL_piecewise125")
#best_pw = min(sweep_pw, key=lambda r: r["val_rmse"])
#print(f"\nBest (piecewise): window={best_pw['window']}, "
    #  f"{best_pw['units1']}→{best_pw['units2']}\n")

#print("=== Sweep — linear label ===")
#sweep_linear = run_lstm_sweep("RUL_linear")
##best_linear = min(sweep_linear, key=lambda r: r["val_rmse"])
#print(f"\nBest (linear): window={best_linear['window']}, "
    #  f"{best_linear['units1']}→{best_linear['units2']}\n")

# --- 3e. Final training on ALL 100 engines with best configs ---
#def final_train_and_eval(best_cfg, label_col, label_name):
   # w = best_cfg["window"]
   # X_tr_full, y_tr_full = build_train_sequences(train, w, label_col)
  #  X_te = build_last_window_sequences(test_full, w, test_last["unit"].values)
   # keras.utils.set_random_seed(SEED)
   # model = build_lstm(w, best_cfg["units1"], best_cfg["units2"])
   # es =# keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
          #                             restore_best_weights=True)
   # t0 = time.time()
   #model.fit(X_tr_full, y_tr_full, validation_split=0.1,
        #      epochs=EPOCHS, batch_size=BATCH, callbacks=[es], verbose=0)
    #t_train = time.time() - t0
   # y_pred = model.predict(X_te, verbose=0).ravel()
   # res = evaluate_model(y_true, y_pred,
              #           model_name=f"LSTM (w={w}, {best_cfg['units1']}→{best_cfg['units2']})",
               #          label_name=label_name)
   # res["training_time_sec"] = round(t_train, 2)
   # res["config"] = f"w={w},{best_cfg['units1']}->{best_cfg['units2']}"
   # return res, y_pred, model

#result_piecewise, y_pred_piecewise, lstm_pw = final_train_and_eval(
  ##  best_pw, "RUL_piecewise125", "piecewise125")
#result_linear, y_pred_linear, lstm_lin = final_train_and_eval(
  #  best_linear, "RUL_linear", "linear")

# --- 3f. Ladder summary ---
#LADDER = {
   # "LR":  {"mae": 17.79, "rmse": 22.07, "nasa": 1334.57, "time": 0.0239},
   # "XGB": {"mae": 13.21, "rmse": 17.71, "nasa": 784.53,  "time": 1.5925},
#}
#print("=" * 68)
#print("LADDER SO FAR — piecewise125 label")
#print("=" * 68)
#print(f"{'':<22} {'LR':>10} {'XGBoost':>10} {'LSTM':>10}")
#print("-" * 68)
#print(f"{'MAE':<22} {LADDER['LR']['mae']:>10.2f} {LADDER['XGB']['mae']:>10.2f} {result_piecewise['mae']:>10.2f}")
#print(f"{'RMSE':<22} {LADDER['LR']['rmse']:>10.2f} {LADDER['XGB']['rmse']:>10.2f} {result_piecewise['rmse']:>10.2f}")
#print(f"{'NASA Score':<22} {LADDER['LR']['nasa']:>10.2f} {LADDER['XGB']['nasa']:>10.2f} {result_piecewise['nasa_score']:>10.2f}")
#print(f"{'Train t##)
#print(f"\nLSTM linear label: MAE {result_linear['mae']}, RMSE {result_linear['rmse']}, "
    #  f"NASA {result_linear['nasa_score']}, time {result_linear['training_time_sec']}s")

## Padding and Masking Issue

During LSTM training, TensorFlow raised the following error:

```text
InvalidArgumentError: RNN masks can only be used for right-padded sequences when using cuDNN.
```

The problem occurred because the input sequences were prepared using zero pre-padding together with a Masking layer.

With zero pre-padding, the sequence has the following form:

```text
[0, 0, 0, Cycle 1, Cycle 2, Cycle 3]
```

The zeros are added at the beginning of the sequence.

However, the cuDNN implementation used by TensorFlow only supports masking when the padding is placed at the end of the sequence:

```text
[Cycle 1, Cycle 2, Cycle 3, 0, 0, 0]
```

Therefore, zero pre-padding, masking, and cuDNN cannot be used together.

## Possible Solutions

### Solution 1 — Keep Zero Pre-padding and Remove Masking

The first solution is to keep the zeros at the beginning of the sequence and remove the Masking layer.

```text
Zero pre-padding
No Masking
cuDNN enabled
```

This solution allows TensorFlow to use the fast GPU-based cuDNN implementation.

The padded zeros are processed as normal input values. Their effect is expected to be limited because they appear only at the beginning of early-life sequences, while the real sensor measurements appear afterward.

### Solution 2 — Use Right-padding with Masking

The second solution is to move the zeros to the end of each sequence.

```text
[Cycle 1, Cycle 2, Cycle 3, 0, 0, 0]
```

The Masking layer can then ignore the padded timesteps, and cuDNN can still be used.

However, right-padding is less suitable for this RUL prediction task because the most recent real cycle would no longer be located at the final timestep of the sequence.

### Solution 3 — Keep Zero Pre-padding and Masking but Disable cuDNN

The third solution is to keep both zero pre-padding and masking while forcing the LSTM layer not to use cuDNN.

```python
LSTM(
    64,
    return_sequences=True,
    use_cudnn=False
)
```

This configuration supports masks with pre-padded sequences.

However, training may become slower because the optimized GPU implementation is disabled.

## Selected Solution

This notebook uses zero pre-padding without a Masking layer.

```text
Padding: Zero pre-padding
Masking: Disabled
Backend: TensorFlow with cuDNN
```

This choice keeps the most recent cycle at the end of each sequence and maintains GPU training speed.

The final configuration is therefore:

```text
Zero pre-padding + No Masking
```


In [ ]:
# ============================================================
# 05_LSTM.ipynb
# Section 3 — Validation Split, Bounded Sweep, Final Training
# Purpose: Same protocol as XGBoost (Notebook 04): engine-level
#          20-engine validation (seed 42), truncation (seed 123),
#          6-config sweep, selection on val RMSE, final train on
#          all engines, single test evaluation per label.
# Padding: zero PRE-padding WITHOUT Masking layer (Option A) —
#          cuDNN-compatible, standard practice in C-MAPSS literature.
# ============================================================

# --- 3a. Engine-level validation split (IDENTICAL to Notebook 04) ---
rng = np.random.RandomState(42)
all_units = train["unit"].unique()
val_units = rng.choice(all_units, size=20, replace=False)
train_units = np.array([u for u in all_units if u not in val_units])

train_sub = train[train["unit"].isin(train_units)]
val_sub   = train[train["unit"].isin(val_units)]

print(f"Training engines:   {len(train_units)}  ({len(train_sub)} rows)")
print(f"Validation engines: {len(val_units)}")
print(f"Validation units:   {sorted(val_units.tolist())}")
print("(Must match Notebook 04: [1, 5, 11, 13, 19, 23, 31, 32, 34, 40, "
      "45, 46, 54, 71, 74, 77, 78, 81, 84, 91])\n")

# --- 3b. Truncated validation engines (IDENTICAL protocol, seed 123) ---
def make_truncation_points(val_sub, rng, min_keep=30):
    """Random truncation cycle per validation engine (same as NB04)."""
    cuts = {}
    for unit, grp in val_sub.groupby("unit"):
        c_max = grp["cycle"].max()
        cuts[unit] = (rng.randint(min_keep, c_max), c_max)
    return cuts

cuts = make_truncation_points(val_sub, np.random.RandomState(123))
val_units_sorted = sorted(cuts)
y_val_true = np.array([cuts[u][1] - cuts[u][0] for u in val_units_sorted],
                      dtype=float)
print(f"Truncated validation: {len(cuts)} engines, "
      f"RUL range {y_val_true.min():.0f}–{y_val_true.max():.0f}\n")

def build_val_windows(val_sub, cuts, window, units_order):
    """Last `window` cycles up to each engine's truncation cycle."""
    X_list = []
    for unit in units_order:
        cut, _ = cuts[unit]
        grp = val_sub[(val_sub["unit"] == unit) &
                      (val_sub["cycle"] <= cut)].sort_values("cycle")
        feats = grp[FINAL_FEATURES_FD001].values
        seq = feats[-window:]
        if len(seq) < window:
            seq = np.vstack([np.zeros((window - len(seq), N_FEATURES)), seq])
        X_list.append(seq)
    return np.array(X_list, dtype=np.float32)

# --- 3c. Model builder (Option A: no Masking layer — cuDNN-compatible) ---
def build_lstm(window, units1, units2, lr=0.001, dropout=0.2):
    """LSTM(units1) -> Dropout -> LSTM(units2) -> Dropout -> Dense(1).
    Contracting form. Zero pre-padding WITHOUT masking (standard
    practice in the C-MAPSS literature; cuDNN-compatible)."""
    model = keras.Sequential([
        layers.Input(shape=(window, N_FEATURES)),
        layers.LSTM(units1, return_sequences=True),
        layers.Dropout(dropout),
        layers.LSTM(units2),
        layers.Dropout(dropout),
        layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss="mse")
    return model

# --- 3d. The sweep: 3 windows x 2 architectures ---
SWEEP = [
    {"window": 15, "units1": 64,  "units2": 32},
    {"window": 30, "units1": 64,  "units2": 32},
    {"window": 50, "units1": 64,  "units2": 32},
    {"window": 15, "units1": 128, "units2": 64},
    {"window": 30, "units1": 128, "units2": 64},
    {"window": 50, "units1": 128, "units2": 64},
]
EPOCHS, BATCH, PATIENCE = 50, 64, 5

def run_lstm_sweep(label_col):
    """Train each config on train_sub; select on truncated-val RMSE."""
    results = []
    # Pre-build sequences once per window size (expensive step)
    seq_cache = {}
    for w in sorted({cfg["window"] for cfg in SWEEP}):
        X_tr, y_tr = build_train_sequences(train_sub, w, label_col)
        X_v = build_val_windows(val_sub, cuts, w, val_units_sorted)
        seq_cache[w] = (X_tr, y_tr, X_v)
        print(f"  [window={w}] train sequences: {X_tr.shape}")
    print()
    for cfg in SWEEP:
        w = cfg["window"]
        X_tr, y_tr, X_v = seq_cache[w]
        keras.utils.set_random_seed(SEED)          # reset seed per config
        model = build_lstm(w, cfg["units1"], cfg["units2"])
        es = keras.callbacks.EarlyStopping(monitor="val_loss",
                                           patience=PATIENCE,
                                           restore_best_weights=True)
        t0 = time.time()
        hist = model.fit(X_tr, y_tr,
                         validation_split=0.1,      # internal early stopping
                         epochs=EPOCHS, batch_size=BATCH,
                         callbacks=[es], verbose=0)
        t_train = time.time() - t0
        pred_v = model.predict(X_v, verbose=0).ravel()
        rmse_v = np.sqrt(mean_squared_error(y_val_true, pred_v))
        n_ep = len(hist.history["loss"])
        results.append({**cfg, "val_rmse": rmse_v,
                        "epochs_run": n_ep, "train_time_s": t_train})
        print(f"  window={w:>2}, {cfg['units1']}→{cfg['units2']:<3} "
              f"->  val RMSE {rmse_v:6.2f}  ({n_ep} epochs, {t_train:.0f}s)")
    return results

print("=== Sweep — piecewise125 label ===")
sweep_pw = run_lstm_sweep("RUL_piecewise125")
best_pw = min(sweep_pw, key=lambda r: r["val_rmse"])
print(f"\nBest (piecewise): window={best_pw['window']}, "
      f"{best_pw['units1']}→{best_pw['units2']}\n")

print("=== Sweep — linear label ===")
sweep_linear = run_lstm_sweep("RUL_linear")
best_linear = min(sweep_linear, key=lambda r: r["val_rmse"])
print(f"\nBest (linear): window={best_linear['window']}, "
      f"{best_linear['units1']}→{best_linear['units2']}\n")

# --- 3e. Final training on ALL 100 engines with best configs ---
def final_train_and_eval(best_cfg, label_col, label_name):
    w = best_cfg["window"]
    X_tr_full, y_tr_full = build_train_sequences(train, w, label_col)
    X_te = build_last_window_sequences(test_full, w, test_last["unit"].values)
    keras.utils.set_random_seed(SEED)
    model = build_lstm(w, best_cfg["units1"], best_cfg["units2"])
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True)
    t0 = time.time()
    model.fit(X_tr_full, y_tr_full, validation_split=0.1,
              epochs=EPOCHS, batch_size=BATCH, callbacks=[es], verbose=0)
    t_train = time.time() - t0
    y_pred = model.predict(X_te, verbose=0).ravel()
    res = evaluate_model(y_true, y_pred,
                         model_name=f"LSTM (w={w}, {best_cfg['units1']}→{best_cfg['units2']})",
                         label_name=label_name)
    res["training_time_sec"] = round(t_train, 2)
    res["config"] = f"w={w},{best_cfg['units1']}->{best_cfg['units2']}"
    return res, y_pred, model

result_piecewise, y_pred_piecewise, lstm_pw = final_train_and_eval(
    best_pw, "RUL_piecewise125", "piecewise125")
result_linear, y_pred_linear, lstm_lin = final_train_and_eval(
    best_linear, "RUL_linear", "linear")

# --- 3f. Ladder summary ---
LADDER = {
    "LR":  {"mae": 17.79, "rmse": 22.07, "nasa": 1334.57, "time": 0.0239},
    "XGB": {"mae": 13.21, "rmse": 17.71, "nasa": 784.53,  "time": 1.5925},
}
print("=" * 68)
print("LADDER SO FAR — piecewise125 label")
print("=" * 68)
print(f"{'':<22} {'LR':>10} {'XGBoost':>10} {'LSTM':>10}")
print("-" * 68)
print(f"{'MAE':<22} {LADDER['LR']['mae']:>10.2f} {LADDER['XGB']['mae']:>10.2f} {result_piecewise['mae']:>10.2f}")
print(f"{'RMSE':<22} {LADDER['LR']['rmse']:>10.2f} {LADDER['XGB']['rmse']:>10.2f} {result_piecewise['rmse']:>10.2f}")
print(f"{'NASA Score':<22} {LADDER['LR']['nasa']:>10.2f} {LADDER['XGB']['nasa']:>10.2f} {result_piecewise['nasa_score']:>10.2f}")
print(f"{'Train time (s)':<22} {LADDER['LR']['time']:>10.4f} {LADDER['XGB']['time']:>10.4f} {result_piecewise['training_time_sec']:>10.2f}")
print("=" * 68)
print(f"\nLSTM linear label: MAE {result_linear['mae']}, RMSE {result_linear['rmse']}, "
      f"NASA {result_linear['nasa_score']}, time {result_linear['training_time_sec']}s")

Notebook 05, first sweep attempt failed: all configs collapsed to predicting the label mean (identical val RMSE ≈ 50.57 across window sizes; test MAE 35.90 ≈ error of constant prediction at label mean 86.8). Cause: raw RUL targets (0–125) vs tanh-bounded LSTM outputs with lr=0.001 — output layer stuck on the mean plateau, early stopping (patience 5) fired before input-dependence was learned. Fix: labels scaled to [0,1] during training (y/y_max, predictions rescaled), patience raised to 10. Standard practice; to be documented in Methodology.



# LSTM Training Issue Documentation

## Problem Description

The initial LSTM sweep produced nearly identical validation results across all tested configurations.

```text
Window 15 → Validation RMSE ≈ 50.57
Window 30 → Validation RMSE ≈ 50.57
Window 50 → Validation RMSE ≈ 50.57
```

Similarly, the larger architecture (128→64) produced nearly identical validation errors. In addition, all configurations terminated after almost the same number of training epochs.

These observations indicate that the LSTM was not learning meaningful temporal relationships from the sensor sequences.

---

## Evidence

Several observations suggest that the network failed to learn from the input sequences.

### Identical validation performance

Different window sizes should provide different temporal information and therefore produce different validation errors.

Instead, all window sizes resulted in nearly identical RMSE values, indicating that changing the input sequence length had almost no effect on the model's predictions.

### Identical training behavior

The number of training epochs was also nearly identical across all configurations.

This suggests that the optimization process followed almost the same trajectory regardless of the selected window size or network architecture.

### Constant prediction behavior

The obtained MAE and RMSE values were consistent with a model that predicts approximately the average RUL value for every sample.

Rather than learning the relationship between sensor measurements and Remaining Useful Life (RUL), the network converged to a nearly constant prediction.

---

## Cause of the Problem

The input features and target labels were represented on significantly different numerical scales.

```text
Input features (normalized):   0 → 1
Target labels (RUL):           0 → 125
```

Although the sensor values were normalized using MinMax scaling, the target RUL values remained in their original range.

This large difference in scale made the optimization process difficult during the early stages of training.

Instead of learning the complex mapping between sensor sequences and RUL, the network converged toward predicting the average target value, which provides a relatively low Mean Squared Error (MSE) compared with random predictions.

Because the validation loss improved only slightly after reaching this constant-prediction state, the training process terminated before the network began learning meaningful temporal degradation patterns.

---

## Impact

As a consequence of this issue:

* The input sequences had little influence on the predictions.
* Different window sizes produced nearly identical validation performance.
* Different network architectures showed almost identical behavior.
* The model failed to learn meaningful degradation patterns from the sensor data.
* The reported evaluation metrics reflected constant predictions rather than successful sequence learning.


In [ ]:
# ============================================================
# 05_LSTM.ipynb
# Section 3 — Validation Split, Bounded Sweep, Final Training
# Protocol identical to Notebook 04. Option A padding (no mask).
# FIX v2: labels scaled to [0,1] during training (y / y_max),
#         predictions rescaled back; patience 5 -> 10.
#         Reason: raw RUL targets (0-125/0-361) vs tanh-bounded
#         LSTM outputs cause mean-collapse with lr=0.001.
# ============================================================

# --- 3a. Engine-level validation split (IDENTICAL to Notebook 04) ---
rng = np.random.RandomState(42)
all_units = train["unit"].unique()
val_units = rng.choice(all_units, size=20, replace=False)
train_units = np.array([u for u in all_units if u not in val_units])

train_sub = train[train["unit"].isin(train_units)]
val_sub   = train[train["unit"].isin(val_units)]

print(f"Training engines:   {len(train_units)}  ({len(train_sub)} rows)")
print(f"Validation engines: {len(val_units)}")
print(f"Validation units:   {sorted(val_units.tolist())}\n")

# --- 3b. Truncated validation engines (seed 123, same as NB04) ---
def make_truncation_points(val_sub, rng, min_keep=30):
    cuts = {}
    for unit, grp in val_sub.groupby("unit"):
        c_max = grp["cycle"].max()
        cuts[unit] = (rng.randint(min_keep, c_max), c_max)
    return cuts

cuts = make_truncation_points(val_sub, np.random.RandomState(123))
val_units_sorted = sorted(cuts)
y_val_true = np.array([cuts[u][1] - cuts[u][0] for u in val_units_sorted],
                      dtype=float)
print(f"Truncated validation: {len(cuts)} engines, "
      f"RUL range {y_val_true.min():.0f}–{y_val_true.max():.0f}\n")

def build_val_windows(val_sub, cuts, window, units_order):
    X_list = []
    for unit in units_order:
        cut, _ = cuts[unit]
        grp = val_sub[(val_sub["unit"] == unit) &
                      (val_sub["cycle"] <= cut)].sort_values("cycle")
        feats = grp[FINAL_FEATURES_FD001].values
        seq = feats[-window:]
        if len(seq) < window:
            seq = np.vstack([np.zeros((window - len(seq), N_FEATURES)), seq])
        X_list.append(seq)
    return np.array(X_list, dtype=np.float32)

# --- 3c. Model builder (Option A: no Masking — cuDNN-compatible) ---
def build_lstm(window, units1, units2, lr=0.001, dropout=0.2):
    model = keras.Sequential([
        layers.Input(shape=(window, N_FEATURES)),
        layers.LSTM(units1, return_sequences=True),
        layers.Dropout(dropout),
        layers.LSTM(units2),
        layers.Dropout(dropout),
        layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss="mse")
    return model

# --- 3d. Sweep: 3 windows x 2 architectures ---
SWEEP = [
    {"window": 15, "units1": 64,  "units2": 32},
    {"window": 30, "units1": 64,  "units2": 32},
    {"window": 50, "units1": 64,  "units2": 32},
    {"window": 15, "units1": 128, "units2": 64},
    {"window": 30, "units1": 128, "units2": 64},
    {"window": 50, "units1": 128, "units2": 64},
]
EPOCHS, BATCH, PATIENCE = 50, 64, 10          # FIX: patience 5 -> 10

def run_lstm_sweep(label_col):
    y_max = float(train_sub[label_col].max())   # FIX: label scale factor
    print(f"  (label scaling: y / {y_max:.0f})")
    results = []
    seq_cache = {}
    for w in sorted({cfg["window"] for cfg in SWEEP}):
        X_tr, y_tr = build_train_sequences(train_sub, w, label_col)
        X_v = build_val_windows(val_sub, cuts, w, val_units_sorted)
        seq_cache[w] = (X_tr, y_tr, X_v)
        print(f"  [window={w}] train sequences: {X_tr.shape}")
    print()
    for cfg in SWEEP:
        w = cfg["window"]
        X_tr, y_tr, X_v = seq_cache[w]
        keras.utils.set_random_seed(SEED)
        model = build_lstm(w, cfg["units1"], cfg["units2"])
        es = keras.callbacks.EarlyStopping(monitor="val_loss",
                                           patience=PATIENCE,
                                           restore_best_weights=True)
        t0 = time.time()
        hist = model.fit(X_tr, y_tr / y_max,               # FIX: scaled labels
                         validation_split=0.1,
                         epochs=EPOCHS, batch_size=BATCH,
                         callbacks=[es], verbose=0)
        t_train = time.time() - t0
        pred_v = model.predict(X_v, verbose=0).ravel() * y_max   # FIX: rescale
        rmse_v = np.sqrt(mean_squared_error(y_val_true, pred_v))
        n_ep = len(hist.history["loss"])
        results.append({**cfg, "val_rmse": rmse_v,
                        "epochs_run": n_ep, "train_time_s": t_train})
        print(f"  window={w:>2}, {cfg['units1']}→{cfg['units2']:<3} "
              f"->  val RMSE {rmse_v:6.2f}  ({n_ep} epochs, {t_train:.0f}s)")
    return results

print("=== Sweep — piecewise125 label ===")
sweep_pw = run_lstm_sweep("RUL_piecewise125")
best_pw = min(sweep_pw, key=lambda r: r["val_rmse"])
print(f"\nBest (piecewise): window={best_pw['window']}, "
      f"{best_pw['units1']}→{best_pw['units2']}\n")

print("=== Sweep — linear label ===")
sweep_linear = run_lstm_sweep("RUL_linear")
best_linear = min(sweep_linear, key=lambda r: r["val_rmse"])
print(f"\nBest (linear): window={best_linear['window']}, "
      f"{best_linear['units1']}→{best_linear['units2']}\n")

# --- 3e. Final training on ALL 100 engines with best configs ---
def final_train_and_eval(best_cfg, label_col, label_name):
    w = best_cfg["window"]
    y_max = float(train[label_col].max())                  # FIX: scale factor
    X_tr_full, y_tr_full = build_train_sequences(train, w, label_col)
    X_te = build_last_window_sequences(test_full, w, test_last["unit"].values)
    keras.utils.set_random_seed(SEED)
    model = build_lstm(w, best_cfg["units1"], best_cfg["units2"])
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True)
    t0 = time.time()
    model.fit(X_tr_full, y_tr_full / y_max,                # FIX: scaled labels
              validation_split=0.1,
              epochs=EPOCHS, batch_size=BATCH, callbacks=[es], verbose=0)
    t_train = time.time() - t0
    y_pred = model.predict(X_te, verbose=0).ravel() * y_max  # FIX: rescale
    res = evaluate_model(y_true, y_pred,
                         model_name=f"LSTM (w={w}, {best_cfg['units1']}→{best_cfg['units2']})",
                         label_name=label_name)
    res["training_time_sec"] = round(t_train, 2)
    res["config"] = f"w={w},{best_cfg['units1']}->{best_cfg['units2']}"
    return res, y_pred, model

result_piecewise, y_pred_piecewise, lstm_pw = final_train_and_eval(
    best_pw, "RUL_piecewise125", "piecewise125")
result_linear, y_pred_linear, lstm_lin = final_train_and_eval(
    best_linear, "RUL_linear", "linear")

# --- 3f. Ladder summary ---
LADDER = {
    "LR":  {"mae": 17.79, "rmse": 22.07, "nasa": 1334.57, "time": 0.0239},
    "XGB": {"mae": 13.21, "rmse": 17.71, "nasa": 784.53,  "time": 1.5925},
}
print("=" * 68)
print("LADDER SO FAR — piecewise125 label")
print("=" * 68)
print(f"{'':<22} {'LR':>10} {'XGBoost':>10} {'LSTM':>10}")
print("-" * 68)
print(f"{'MAE':<22} {LADDER['LR']['mae']:>10.2f} {LADDER['XGB']['mae']:>10.2f} {result_piecewise['mae']:>10.2f}")
print(f"{'RMSE':<22} {LADDER['LR']['rmse']:>10.2f} {LADDER['XGB']['rmse']:>10.2f} {result_piecewise['rmse']:>10.2f}")
print(f"{'NASA Score':<22} {LADDER['LR']['nasa']:>10.2f} {LADDER['XGB']['nasa']:>10.2f} {result_piecewise['nasa_score']:>10.2f}")
print(f"{'Train time (s)':<22} {LADDER['LR']['time']:>10.4f} {LADDER['XGB']['time']:>10.4f} {result_piecewise['training_time_sec']:>10.2f}")
print("=" * 68)
print(f"\nLSTM linear label: MAE {result_linear['mae']}, RMSE {result_linear['rmse']}, "
      f"NASA {result_linear['nasa_score']}, time {result_linear['training_time_sec']}s")

# LSTM Training Fix Documentation

## Implemented Changes

To address the training collapse observed during the first LSTM sweep, several modifications were applied to the training pipeline.

### 1. Target Label Scaling

The target labels were normalized during training.

Instead of training the network directly on raw RUL values,

```text
0 → 125
```

or

```text
0 → 361
```

the labels were scaled to

```text
0 → 1
```

using

```python
y_scaled = y / y_max
```

where `y_max` is the maximum RUL value for the corresponding label.

After inference, the predictions were converted back to the original RUL scale using

```python
y_pred = y_pred_scaled × y_max
```

This removed the large numerical mismatch between the normalized input features and the target values, allowing the optimizer to learn a stable mapping between sensor sequences and RUL.

---

### 2. Early Stopping Adjustment

The Early Stopping patience was increased from

```text
5
```

to

```text
10
```

This provided the network with additional training time before termination, allowing it to move beyond the initial constant-prediction phase and begin learning meaningful temporal patterns.

---

### 3. Zero Pre-padding Without Masking

The sequence preprocessing continued to use zero pre-padding while the Masking layer was removed.

This configuration maintains compatibility with TensorFlow's cuDNN implementation and enables fast GPU training without changing the temporal ordering of the sequences.

---

# Results After the Fix

The modified training pipeline produced clear evidence that the LSTM was successfully learning from the input sequences.

### Validation RMSE became configuration-dependent

Unlike the previous run, the validation RMSE values now varied across different window sizes and architectures.

```text
Window 15 (64→32)   → 26.47
Window 30 (64→32)   → 20.73
Window 50 (64→32)   → 18.46

Window 15 (128→64)  → 26.99
Window 30 (128→64)  → 19.87
Window 50 (128→64)  → 17.37
```

This demonstrates that the model was no longer producing constant predictions and that the temporal information contained in the input windows was influencing the learning process.

---

### Different Training Dynamics

Each configuration stopped after a different number of epochs.

This indicates that each model followed its own optimization trajectory rather than collapsing to the same local solution.

---

### Best Configuration

Both target labels selected the same architecture.

```text
Window = 50
Architecture = 128 → 64
```

This consistency simplifies the final model selection and supports the robustness of the chosen architecture.

---

### Final Performance

For the piecewise125 label, the final LSTM achieved:

```text
MAE        = 10.69 cycles
RMSE       = 14.78 cycles
NASA Score = 402.11
```

Compared with the previous models:

| Model             |       MAE |      RMSE | NASA Score |
| ----------------- | --------: | --------: | ---------: |
| Linear Regression |     17.79 |     22.07 |    1334.57 |
| XGBoost           |     13.21 |     17.71 |     784.53 |
| LSTM              | **10.69** | **14.78** | **402.11** |

The LSTM became the best-performing model across all evaluation metrics, confirming that incorporating temporal information significantly improved Remaining Useful Life prediction.

---

# Conclusion

The implemented modifications successfully resolved the training collapse observed during the initial LSTM experiments.

After applying target label scaling, increasing the Early Stopping patience, and using zero pre-padding without masking, the network learned meaningful temporal degradation patterns instead of converging to constant predictions.

The resulting LSTM model outperformed both Linear Regression and XGBoost, establishing it as the strongest model evaluated in this study.


In [ ]:
# ============================================================
# 05_LSTM.ipynb
# Section 4 — Visualization, Zone Analysis, Bias Test
# ============================================================

# --- 4a. Predicted vs True scatter — both labels ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, y_pred, label_name, res in [
    (axes[0], y_pred_linear,    "Linear RUL label",        result_linear),
    (axes[1], y_pred_piecewise, "Piecewise-125 RUL label", result_piecewise),
]:
    ax.scatter(y_true, y_pred, alpha=0.6, s=35, color="#2E75B6",
               edgecolor="white", linewidth=0.5)
    lims = [0, max(y_true.max(), y_pred.max()) * 1.05]
    ax.plot(lims, lims, "r--", linewidth=1.2, label="Perfect prediction")
    ax.set_xlabel("True RUL (cycles)")
    ax.set_ylabel("Predicted RUL (cycles)")
    ax.set_title(f"LSTM — {label_name}\nMAE {res['mae']:.2f} | "
                 f"RMSE {res['rmse']:.2f} | NASA {res['nasa_score']:.0f}")
    ax.legend(loc="upper left")
    ax.set_xlim(lims); ax.set_ylim(lims)

fig.suptitle("FD001 — LSTM: predicted vs. true RUL (100 test engines, last cycle)",
             fontsize=13, y=1.02)
fig.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fd001_lstm_pred_vs_true.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}\n")

# --- 4b. Error vs True RUL ---
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
for ax, y_pred, label_name in [
    (axes2[0], y_pred_linear,    "Linear RUL label"),
    (axes2[1], y_pred_piecewise, "Piecewise-125 RUL label"),
]:
    errors = y_pred - y_true
    ax.scatter(y_true, errors, alpha=0.6, s=35, color="#D9822B",
               edgecolor="white", linewidth=0.5)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("True RUL (cycles)")
    ax.set_ylabel("Prediction error (pred − true, cycles)")
    ax.set_title(f"{label_name}\nAbove 0 = late (dangerous), below 0 = early (conservative)")

fig2.suptitle("FD001 — LSTM: prediction error vs. true RUL", fontsize=13, y=1.03)
fig2.tight_layout()
fig2_path = os.path.join(FIGURES_DIR, "fd001_lstm_error_vs_true.png")
fig2.savefig(fig2_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig2_path}\n")

# --- 4c. Zone analysis — LSTM vs XGBoost vs LR (piecewise) ---
print("=== Zone analysis — piecewise label, all three models ===\n")
errors_pw = y_pred_piecewise - y_true

BASELINES = {
    "Near failure (RUL <= 30)":  {"XGB": (10.07, +8.59),  "LR": (19.85, +18.77)},
    "Mid life (30 < RUL <= 80)": {"XGB": (14.83, +8.83),  "LR": (19.76, +19.75)},
    "Early life (RUL > 80)":     {"XGB": (14.04, -3.21),  "LR": (16.14, -8.46)},
}
zones = [("Near failure (RUL <= 30)",  y_true <= 30),
         ("Mid life (30 < RUL <= 80)", (y_true > 30) & (y_true <= 80)),
         ("Early life (RUL > 80)",     y_true > 80)]

print(f"{'Zone':<28} {'n':>4} {'LSTM MAE':>9} {'LSTM bias':>10} "
      f"{'XGB bias':>9} {'LR bias':>8}")
print("-" * 74)
for zone_name, mask in zones:
    n = mask.sum()
    z_mae  = np.abs(errors_pw[mask]).mean()
    z_bias = errors_pw[mask].mean()
    xgb_bias = BASELINES[zone_name]["XGB"][1]
    lr_bias  = BASELINES[zone_name]["LR"][1]
    print(f"{zone_name:<28} {n:>4} {z_mae:>9.2f} {z_bias:>+10.2f} "
          f"{xgb_bias:>+9.2f} {lr_bias:>+8.2f}")

# --- 4d. Top NASA contributors — linear label ---
print("\n=== Top 5 NASA-Score contributors — linear label ===\n")
d = y_pred_linear - y_true
scores = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)
top5 = np.argsort(scores)[::-1][:5]
print(f"{'unit':>5} {'true RUL':>9} {'predicted':>10} {'error':>8} {'score contrib':>14}")
print("-" * 52)
for i in top5:
    print(f"{test_last['unit'].values[i]:>5} {y_true[i]:>9.0f} "
          f"{y_pred_linear[i]:>10.2f} {d[i]:>+8.2f} {scores[i]:>14.1f}")
print(f"\nTop-5 share of total NASA Score: "
      f"{100 * scores[top5].sum() / scores.sum():.1f}%")

# LSTM Training Process and Early Stopping Documentation

## Training Procedure

For each configuration in the sweep, a completely new LSTM model is created.

```python
model = build_lstm(...)
```

Each model starts with randomly initialized weights and is trained independently from all previous configurations.

The sweep evaluates six different configurations by combining three sequence lengths with two network architectures.

```text
Window 15 → 64→32
Window 30 → 64→32
Window 50 → 64→32

Window 15 → 128→64
Window 30 → 128→64
Window 50 → 128→64
```

For every configuration, the model is trained using

```python
model.fit(...)
```

which begins the optimization process.

---

# What Happens During Each Epoch

During every epoch, the following sequence of operations is performed.

### Step 1 — Forward Pass

The LSTM receives one batch of training sequences.

Example:

```text
Input sequence (50 cycles)
        ↓
LSTM
        ↓
Prediction
```

Initially, the model weights are random, so the predictions are also nearly random.

For example,

```text
True scaled RUL = 0.76
Predicted RUL   = 0.61
```

The prediction error is computed using the Mean Squared Error (MSE) loss.

---

### Step 2 — Weight Update

The Adam optimizer computes the gradients of the loss with respect to all trainable parameters.

The optimizer then updates the model weights to reduce the prediction error.

After several epochs, the prediction gradually moves closer to the target.

Example:

```text
Epoch 1 → Prediction = 0.61
Epoch 2 → Prediction = 0.69
Epoch 3 → Prediction = 0.73
Epoch 4 → Prediction = 0.75
```

As the predictions improve, the training loss decreases.

---

### Step 3 — Validation

After completing one training epoch, the model is evaluated using the validation subset specified by

```python
validation_split = 0.1
```

The validation data is never used for updating the model weights.

Instead, it is used only to measure the model's ability to generalize to unseen data.

The monitored metric is

```text
Validation Loss
```

rather than the training loss.

---

# Early Stopping Mechanism

The notebook uses

```python
EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)
```

The callback continuously monitors the validation loss after every epoch.

If the validation loss decreases, the model is considered to have improved.

For example,

| Epoch | Validation Loss |
| ----: | --------------: |
|    15 |           0.040 |
|    16 |           0.037 |

Since the validation loss improved, the patience counter is reset to zero.

---

If the validation loss does not improve, the patience counter increases.

Example:

| Epoch | Validation Loss | Counter |
| ----: | --------------: | ------: |
|    17 |           0.037 |       1 |
|    18 |           0.038 |       2 |
|    19 |           0.038 |       3 |
|   ... |             ... |     ... |

Training continues while the counter remains below the patience value.

---

# When Does Early Stopping Stop Training?

Training stops only after

```text
10 consecutive epochs
```

without any improvement in the validation loss.

For example,

| Epoch |              Validation Loss |
| ----: | ---------------------------: |
|    25 | 0.026 ← Best validation loss |
|    26 |                        0.026 |
|    27 |                        0.027 |
|    28 |                        0.027 |
|    29 |                        0.028 |
|    30 |                        0.028 |
|    31 |                        0.029 |
|    32 |                        0.029 |
|    33 |                        0.030 |
|    34 |                        0.030 |
|    35 |                        0.031 |

The best validation performance occurred at Epoch 25.

From Epoch 26 onward, the validation loss never improved.

After ten consecutive epochs without improvement, Early Stopping terminated the training process.

---

# Restoring the Best Model

The parameter

```python
restore_best_weights=True
```

ensures that the final model is not the one obtained in the last training epoch.

Instead, TensorFlow automatically restores the model weights corresponding to the epoch that achieved the lowest validation loss.

In the previous example,

```text
Best epoch = 25
Training stopped = 35
```

Although training continued until Epoch 35, the final saved model corresponds to the weights obtained at Epoch 25 because they produced the best validation performance.

---

# Interpretation of the Training Output

The output

```text
window=50, 128→64 → 30 epochs
```

does not indicate that the model completed all planned epochs.

Instead, it indicates that:

1. The model started training from randomly initialized weights.
2. The validation loss continued improving for several epochs.
3. The best validation performance was achieved before the maximum epoch limit.
4. The validation loss then stopped improving.
5. Early Stopping waited for ten consecutive epochs without improvement.
6. Training was terminated automatically.
7. The weights from the best validation epoch were restored before evaluation.

This strategy prevents unnecessary training while preserving the model that achieved the best generalization performance on the validation data.


In [ ]:
# ============================================================
# 05_LSTM.ipynb
# Section 5 — Save Predictions, Sweep Table, and Finalize
# Purpose: Save per-engine predictions + tuning sweep to results/,
#          print final summary for the experiment log.
# ============================================================

# --- 5a. Per-engine predictions table ---
predictions = pd.DataFrame({
    "unit": test_last["unit"].values,
    "cycle_last": test_last["cycle"].values,
    "RUL_true": y_true,
    "pred_LSTM_linear": np.round(y_pred_linear, 2),
    "pred_LSTM_piecewise125": np.round(y_pred_piecewise, 2),
})
predictions["error_linear"] = np.round(
    predictions["pred_LSTM_linear"] - predictions["RUL_true"], 2)
predictions["error_piecewise125"] = np.round(
    predictions["pred_LSTM_piecewise125"] - predictions["RUL_true"], 2)

print("Per-engine predictions (first 10 rows):")
print(predictions.head(10).to_string(index=False))

pred_path = os.path.join(RESULTS_DIR, "fd001_lstm_predictions.csv")
predictions.to_csv(pred_path, index=False)
print(f"\nSaved predictions to:\n{pred_path}")

# --- 5b. Save the tuning sweep table (both labels) ---
sweep_rows = []
for r in sweep_pw:
    sweep_rows.append({"window": r["window"], "units1": r["units1"],
                       "units2": r["units2"], "label": "piecewise125",
                       "val_rmse": round(r["val_rmse"], 2),
                       "epochs_run": r["epochs_run"],
                       "train_time_s": round(r["train_time_s"], 1)})
for r in sweep_linear:
    sweep_rows.append({"window": r["window"], "units1": r["units1"],
                       "units2": r["units2"], "label": "linear",
                       "val_rmse": round(r["val_rmse"], 2),
                       "epochs_run": r["epochs_run"],
                       "train_time_s": round(r["train_time_s"], 1)})
sweep_df = pd.DataFrame(sweep_rows)

# Mark the selected config per label
sweep_df["selected"] = False
for label, best in [("piecewise125", best_pw), ("linear", best_linear)]:
    mask = ((sweep_df["label"] == label) &
            (sweep_df["window"] == best["window"]) &
            (sweep_df["units1"] == best["units1"]) &
            (sweep_df["units2"] == best["units2"]))
    sweep_df.loc[mask, "selected"] = True

sweep_path = os.path.join(RESULTS_DIR, "fd001_lstm_tuning_sweep.csv")
sweep_df.to_csv(sweep_path, index=False)
print(f"\nSaved tuning sweep to:\n{sweep_path}\n")
print("Tuning sweep table:")
print(sweep_df.to_string(index=False))

# --- 5c. Final summary block ---
print("\n" + "=" * 60)
print("NOTEBOOK 05 COMPLETE — Final Results Summary")
print("=" * 60)
print(f"""
Model: LSTM (Keras/TensorFlow, GPU T4)
Architecture: 2-layer contracting, selected via 6-config sweep
Best config (both labels): window=50, 128→64
Training details: label-scaled targets (y/y_max), zero pre-padding
without masking, dropout 0.2, Adam lr=0.001, batch 64,
early stopping patience 10, seed 42.
Test protocol: last available cycle per test engine (n=100)

Run 1 — linear label (w=50, 128→64):
  MAE {result_linear['mae']}, RMSE {result_linear['rmse']}, NASA {result_linear['nasa_score']}, time {result_linear['training_time_sec']}s

Run 2 — piecewise125 label (w=50, 128→64):
  MAE {result_piecewise['mae']}, RMSE {result_piecewise['rmse']}, NASA {result_piecewise['nasa_score']}, time {result_piecewise['training_time_sec']}s

Files saved:
  results/fd001_lstm_predictions.csv
  results/fd001_lstm_tuning_sweep.csv
  figures/fd001_lstm_pred_vs_true.png
  figures/fd001_lstm_error_vs_true.png
""")
print("=" * 60)

# Notebook 05 — LSTM: Complete Documentation

## Purpose

Train and evaluate the third model of the complexity ladder: a Long
Short-Term Memory (LSTM) network — the first model on the ladder that
sees temporal context. Instead of a single-cycle snapshot, the LSTM
receives a sliding window of the last W cycles (selected: W=50) and can
therefore infer not just the current degradation state but also the rate
and pattern of degradation.

**Position in the project.** Third of four modeling notebooks
(03 Linear Regression → 04 XGBoost → 05 LSTM → 06 Chronos). Identical
frozen data from `02_Preprocessing.ipynb`, identical evaluation protocol,
identical validation split and tuning budget as XGBoost.

**Why LSTM as step 3.** XGBoost (Notebook 04) halved the dangerous
near-failure over-prediction (+18.77 → +8.59) but could not eliminate
it. The hypothesized cause: temporal blindness. XGBoost sees single
rows — it cannot observe how fast an engine is degrading. The EDA
(Section 11) showed degradation rates vary by 3.3× across engines, so
identical sensor snapshots can correspond to very different remaining
lives. The LSTM processes sequences and can resolve this. The testable
hypothesis of this notebook: temporal context reduces the remaining
near-failure bias.

## Inputs and outputs

**Inputs (from `data/processed/`):**
- `train_FD001_processed.parquet` (20631 rows, 13 features + 2 RUL labels)
- `test_FD001_processed.parquet` (13096 rows — full test trajectories,
  needed to build the input windows)
- `test_FD001_last_cycle.parquet` (100 rows, RUL_true)

**Outputs:**
- `results/fd001_lstm_predictions.csv` — per-engine predictions, both labels
- `results/fd001_lstm_tuning_sweep.csv` — 12-row sweep table
- `figures/fd001_lstm_pred_vs_true.png`
- `figures/fd001_lstm_error_vs_true.png`
- Experiment log rows EXP-005 (linear) and EXP-006 (piecewise125)

## Section overview

**Section 1 — Setup, data loading, GPU check.** First notebook on the
ladder requiring GPU (T4). Seeds fixed (42); GPU training retains minor
non-determinism, documented.

**Section 2 — Metrics + sequence construction.** Identical metric
functions as Notebooks 03/04. New: sliding-window builders. Each
training sample = window of the last W cycles (shape W×13), label = RUL
at the window's last cycle. Early cycles zero-PRE-padded (most recent
cycle always at the sequence end). Alignment and padding verified by
assertions: one sample per training row (20631), window's last cycle ↔
label at that cycle, test windows aligned with the last-cycle protocol.

**Section 3 — Validation split, bounded sweep, final training.**
Identical protocol to XGBoost: same 20 validation engines (seed 42),
same truncation protocol (seed 123), 6-config sweep (windows 15/30/50 ×
architectures 64→32 / 128→64), selection on validation RMSE, final
training on all 100 engines, one test evaluation per label.

**Two implementation lessons (documented for Methodology):**
1. *cuDNN masking constraint:* TensorFlow's GPU LSTM (cuDNN) does not
   support masks with pre-padded sequences. Decision (Option A): zero
   pre-padding WITHOUT a Masking layer — standard practice in the
   C-MAPSS literature, keeps full GPU speed. Affected samples are
   early-life windows only, where the piecewise label is flat at 125.
2. *Label scaling:* the first sweep attempt collapsed to predicting the
   label mean (identical val RMSE ≈ 50.6 across all configs; test MAE
   35.90 ≈ error of a constant prediction at the label mean 86.8).
   Cause: raw RUL targets (0–125/0–361) vs tanh-bounded LSTM outputs
   with lr=0.001 — the output layer sat on the mean plateau and early
   stopping (patience 5) fired before input-dependence was learned.
   Fix: labels scaled to [0,1] during training (y/y_max), predictions
   rescaled; patience raised to 10. Standard practice.

**Section 4 — Visualization and zone analysis.** Same diagnostics as
Notebooks 03/04, plus the three-model zone comparison.

**Section 5 — Save and finalize.**

## Sweep results (validation RMSE)

| Window | Architecture | Piecewise | Linear |
|---|---|---|---|
| 15 | 64→32 | 26.47 | 31.65 |
| 30 | 64→32 | 20.73 | 20.73 |
| 50 | 64→32 | 18.46 | 17.45 |
| 15 | 128→64 | 26.99 | 31.78 |
| 30 | 128→64 | 19.87 | 20.07 |
| **50** | **128→64** | **17.37 ← selected** | **17.22 ← selected** |

**Sweep finding — window size matters on FD001 (below 50):** validation
RMSE improves monotonically with window length (15 → 30 → 50). This
refines the literature: Guilherme (content.pdf) reported insensitivity
between windows 50 and 60; our sweep shows strong sensitivity below 50.
Together: FD001 needs a sufficiently long window (~50 cycles); beyond
that, returns flatten. Both labels independently selected the same
configuration (w=50, 128→64).

## Results (recorded, final run)

| Run | Label | MAE | RMSE | NASA Score | Train time (s) |
|---|---|---|---|---|---|
| EXP-005 | linear | 18.30 | 25.73 | 23146.52 | 49.55 |
| EXP-006 | piecewise125 | 10.69 | 14.78 | 402.11 | 80.74 |

**LSTM (piecewise) vs. XGBoost (piecewise):**
- MAE: 13.21 → 10.69 (−19.1%)
- RMSE: 17.71 → 14.78 (−16.5%)
- NASA Score: 784.53 → 402.11 (−48.7%)
- Training time: 1.59 s (CPU) → 80.74 s (GPU T4) — ~51× in wall-clock
  terms; hardware differs and is documented as such.

## Zone analysis — the trajectory hypothesis confirmed

| Zone | n | LSTM MAE | LSTM bias | XGB bias | LR bias |
|---|---|---|---|---|---|
| Near failure (RUL ≤ 30) | 25 | 2.72 | **+0.24** | +8.59 | +18.77 |
| Mid life (30 < RUL ≤ 80) | 20 | 11.80 | +5.58 | +8.83 | +19.75 |
| Early life (RUL > 80) | 55 | 13.91 | −2.34 | −3.21 | −8.46 |

**The central finding of the ladder so far.** The dangerous near-failure
over-prediction bias falls from +18.77 (LR) to +8.59 (XGBoost) to
+0.24 (LSTM) — essentially zero. The LSTM's near-failure MAE is 2.72
cycles: engines about to fail are located to within ~3 cycles on
average. The hypothesis chain is confirmed end-to-end:
- LR's bias was caused partly by linearity (fixed by XGBoost) and
- partly by temporal blindness (fixed by the LSTM reading 50-cycle
  trajectories and inferring degradation rate).

## Complexity-label interaction — third confirmation

Linear label: NASA Score 23,147 (piecewise: 402 — 58× better). Five
engines account for 94.7% of the linear-label score; engine 67 alone
(true RUL 77, predicted 176) contributes ~19,000. Pattern across the
ladder: LR 14k → XGBoost 99k → LSTM 23k. Every flexible model amplifies
the linear label's flaws; the piecewise label is not optional at higher
model complexity.

## Strengths (Vorteile)

1. **Best accuracy on the ladder.** MAE 10.69, RMSE 14.78, NASA 402 —
   beats XGBoost on every metric, competitive with published plain-LSTM
   results on FD001 (literature RMSE ≈ 16).
2. **Near-failure bias eliminated.** +0.24 mean error in the critical
   zone; near-failure MAE 2.72 cycles. The operationally most valuable
   property of any model on this ladder.
3. **Reads degradation rate.** The only model so far that resolves the
   3.3× rate heterogeneity — two engines with identical current
   readings but different trajectories get different predictions.
4. **Window sweep produced a citable finding** (window sensitivity
   below 50 on FD001, refining the literature).

## Weaknesses (Nachteile)

1. **Highest training cost so far.** 80.7 s on a GPU vs 1.6 s (CPU) for
   XGBoost — and requires GPU hardware at all. In wall-clock terms ~51×
   XGBoost; the cost ladder steepens sharply.
2. **Not interpretable.** No feature importance, no readable weights.
   XGBoost could show *which* sensors drive predictions; the LSTM
   cannot (without additional attribution methods, out of scope).
3. **Fragile training dynamics.** The mean-collapse failure showed that
   the LSTM requires careful target scaling and early-stopping settings;
   LR and XGBoost have no such failure modes. This operational
   fragility is itself a cost of complexity.
4. **Non-deterministic on GPU.** Results vary slightly between runs even
   with fixed seeds; documented.
5. **Mid-life zone least improved.** Bias +5.58 remains (XGB +8.83).
   The mid-life region — where degradation has begun but trends are
   still weak — stays the hardest zone for every model.

## The precise problem statement for Notebook 06

> The LSTM learned everything it knows from 100 training engines of one
> dataset. Its knowledge of "how time series behave" is entirely
> task-specific. A time-series foundation model (Chronos) has been
> pretrained on millions of diverse time series and arrives with general
> temporal knowledge. The question for the final ladder step: does this
> pretrained general knowledge add anything on a simple, single-condition
> task where a task-trained LSTM already performs strongly — and at what
> computational cost? Notably, Chronos is a univariate forecaster and
> cannot natively consume 13 sensor channels — the adaptation itself is
> part of the cost.

## Handoff to Chronos (Notebook 06)

| Aspect | LSTM | Chronos |
|---|---|---|
| Temporal context | ✅ learned from this dataset | ✅ pretrained on millions of series |
| Task-specific training | full training required | zero-shot or light adaptation |
| Multivariate input | native (13 channels) | ❌ univariate — needs adaptation strategy |
| Expected cost | 80 s training | inference-heavy; potentially minutes |

**Key questions for Notebook 06:** (1) can a pretrained univariate
forecaster compete with a task-trained multivariate LSTM on FD001?
(2) what is the cost profile (no training, but heavy inference)?
(3) how is the multivariate→univariate gap bridged, and what is lost?

## Ladder so far

| Model | MAE (piecewise) | NASA (piecewise) | Near-failure bias | Cost |
|---|---|---|---|---|
| Linear Regression | 17.79 | 1334.57 | +18.77 | 0.02 s CPU |
| XGBoost (tuned) | 13.21 | 784.53 | +8.59 | 1.59 s CPU |
| LSTM (tuned) | **10.69** | **402.11** | **+0.24** | 80.74 s GPU |
| Chronos | — | — | — | — |

**Next:** `06_Chronos.ipynb` — the final rung: a pretrained time-series
foundation model. Go/no-go decision date: August 10.

# Why We Selected Chronos After LSTM

Although the LSTM achieved the best performance so far in this thesis, it still has several important limitations. These limitations motivated the final step of the complexity ladder: **Chronos**, a pretrained time-series foundation model.

---

# 1. Knowledge is Limited to the Training Dataset

## Limitation

The LSTM learned exclusively from:

- 100 training engines
- The FD001 subset only

Its understanding of degradation is therefore specific to this single dataset.

If any of the following changes:

- Engine type
- Fault mode
- Operating conditions

the model must be retrained from scratch.

## Why Chronos?

Chronos is a **foundation model** pretrained on millions of diverse time series. It already possesses general temporal knowledge that can be transferred to new tasks without learning everything from the beginning.

---

# 2. Training is Fragile and Requires Careful Tuning

## Limitation

To obtain the final LSTM performance, the training required:

- Target (label) scaling
- Early stopping
- Patience tuning
- Fixed random seeds
- GPU hardware
- Solving the mean-collapse problem

Small implementation mistakes could significantly degrade the model's performance.

## Why Chronos?

Chronos is already pretrained. Instead of training an entire deep neural network, only a lightweight regression head needs to be trained, making the training process much simpler and more stable.

---

# 3. High Computational Cost

## Limitation

The LSTM required:

- GPU acceleration
- Approximately 80 seconds of training
- Complete retraining whenever the architecture or hyperparameters changed

## Why Chronos?

Most of the learning has already been completed during pretraining. Only minimal task-specific adaptation is required, greatly reducing training effort.

---

# 4. Non-Deterministic Training

## Limitation

Even with fixed random seeds:

- GPU execution remains slightly non-deterministic.
- Results may vary slightly between runs.

This reduces experiment reproducibility.

## Why Chronos?

Since Chronos relies on a pretrained backbone, it minimizes dependence on long stochastic training procedures.

---

# 5. Task-Specific Rather Than General Knowledge

## Limitation

The LSTM learned only:

- The degradation behavior of FD001.

Its knowledge cannot easily generalize beyond this task.

## Why Chronos?

Chronos has learned temporal patterns from millions of time series, including:

- Long-term trends
- Seasonal behavior
- Gradual drifts
- Sudden changes

This provides much broader temporal knowledge than a task-specific LSTM.

---

# 6. Mid-Life Prediction Remains Difficult

## LSTM Zone Analysis

| Zone | Bias |
|------|------:|
| Near Failure | +0.24 |
| Mid-Life | +5.58 |
| Early Life | -2.34 |

## Limitation

Although the LSTM almost completely eliminated the near-failure bias, it still consistently overestimates RUL in the **Mid-Life** region.

This remains the most difficult prediction zone for all models evaluated in this thesis.

## Why Chronos?

The key research question is whether the broad temporal knowledge learned during Chronos pretraining can improve prediction performance in this challenging region.

---

# 7. Poor Knowledge Transfer

## Limitation

Whenever a new dataset or operating scenario is introduced, the LSTM must be trained again from scratch.

## Why Chronos?

Chronos was specifically designed for transfer learning across multiple time-series tasks, making it much easier to adapt to new domains.

---

# Overall Motivation for Selecting Chronos

Choosing Chronos does **not** imply that the LSTM is a weak model. In fact, the LSTM is the strongest task-specific model evaluated in this thesis.

However, Chronos represents a fundamentally different learning paradigm.

- **LSTM** learns temporal patterns only from the target dataset.
- **Chronos** learns general temporal knowledge from millions of diverse time series before being applied to a specific task.

This motivates the final research question of the thesis:

> Can a pretrained time-series foundation model outperform a task-trained LSTM for Remaining Useful Life prediction while requiring less task-specific training and providing more general temporal knowledge?